<a href="https://colab.research.google.com/github/Carlosjara01/Inteligencia-Artificial/blob/main/clasificacion_sentimientos_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Clasificación de Sentimientos con Transformers
**Programación 5 — Modelos Generativos y NLP**

---

## 📚 Marco Teórico

### 1. ¿Qué son los Modelos Generativos?
Los **modelos generativos** son sistemas de IA entrenados sobre grandes cantidades de texto para aprender patrones del lenguaje. A diferencia de los modelos tradicionales que solo clasifican, estos modelos pueden **representar, comprender y generar** texto. BERT, GPT y sus variantes son ejemplos de modelos generativos basados en la arquitectura Transformer.

---

### 2. ¿Qué es un Transformer?
El **Transformer** es una arquitectura de red neuronal propuesta en 2017 (paper: *"Attention is All You Need"*). Su innovación principal es el mecanismo de **atención**, que permite al modelo relacionar cualquier palabra del texto con cualquier otra, sin importar la distancia entre ellas. Esto supera las limitaciones de las RNN/LSTM que procesaban el texto de forma secuencial.

**Flujo general:**
```
Texto → Tokens → Embeddings → Capas de Atención → Representación → Tarea específica
```

---

### 3. ¿Qué es BERT?
**BERT** (*Bidirectional Encoder Representations from Transformers*) es un modelo preentrenado por Google (2018). Sus características principales:
- **Bidireccional**: lee el texto en ambas direcciones simultáneamente (izquierda→derecha y derecha→izquierda).
- **Preentrenado**: fue entrenado con Wikipedia y BookCorpus (millones de textos).
- **Fine-tuning**: se puede adaptar a tareas específicas como clasificación de sentimientos.

---

### 4. ¿Qué es DistilBERT?
**DistilBERT** es una versión comprimida y acelerada de BERT creada por HuggingFace. Conserva el **97% de la capacidad** de BERT pero es:
- 40% más pequeño
- 60% más rápido
- Ideal para proyectos con recursos limitados (como Google Colab gratuito)

Funciona igual que BERT pero con menos capas (6 en lugar de 12).

---

### 5. ¿Qué son los Embeddings?
Un **embedding** es la representación numérica de una palabra o token como un vector de números reales. Por ejemplo:
- `"feliz"` → `[0.23, -0.41, 0.87, ..., 0.12]` (vector de 768 dimensiones en BERT)

Palabras con significados similares tienen vectores **cercanos en el espacio**. Los embeddings capturan el contexto: la misma palabra puede tener vectores distintos según la oración.

---

### 6. Mecanismo de Atención Q, K, V
El mecanismo de atención funciona con tres matrices:

| Matriz | Nombre | Rol |
|--------|--------|-----|
| **Q** | Query (Consulta) | "¿Qué estoy buscando?" |
| **K** | Key (Clave) | "¿Qué información tengo disponible?" |
| **V** | Value (Valor) | "¿Cuál es el contenido real?" |

**Fórmula:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Cada token "atiende" a todos los demás. El resultado es un vector que combina la información más relevante del contexto.

---

### 7. Promedio Ponderado
Después de la atención, cada token tiene su propio vector enriquecido por el contexto. Para obtener **una sola representación de toda la oración**, se calcula el **promedio de todos los vectores de tokens**. Esto se llama *mean pooling* y produce un único vector de 768 dimensiones que representa el significado global del texto.

---

### 8. Capa Lineal (Linear Layer)
Una **capa lineal** (también llamada capa densa o fully connected) es una transformación matemática:
$$y = xW + b$$
Donde `W` es la matriz de pesos y `b` el sesgo (*bias*). En clasificación de sentimientos, esta capa transforma el vector de 768 dimensiones en **2 valores** (uno por clase: positivo y negativo). El softmax convierte esos valores en probabilidades.

---

### 9. Clasificación de Sentimientos
La **clasificación de sentimientos** (*Sentiment Analysis*) es una tarea de NLP que determina la polaridad emocional de un texto:
- **Positivo**: expresa satisfacción, alegría, aprobación.
- **Negativo**: expresa insatisfacción, tristeza, rechazo.

Con modelos Transformer preentrenados para esta tarea, el modelo ya aprendió los patrones del lenguaje y solo necesita interpretar el texto nuevo.

---

## 🔄 Flujo del Sistema

```
Texto
  │
  ▼
Tokenización ── Texto → lista de IDs numéricos + attention_mask
  │
  ▼
Embedding ──── IDs → vectores de 768 dimensiones por token
  │
  ▼
Atención Q,K,V ── cada token atiende a todos los demás
  │
  ▼
Promedio Ponderado ── un solo vector representa toda la oración
  │
  ▼
Capa Lineal ── vector 768D → logits [neg_score, pos_score]
  │
  ▼
Softmax ── logits → probabilidades [0.0 a 1.0]
  │
  ▼
Resultado: POSITIVO / NEGATIVO con % de confianza
```

---
## ⚙️ PASO 1: Instalación de librerías
Instalamos `transformers` de HuggingFace y `torch` (PyTorch) para trabajar con modelos preentrenados.

In [1]:
# ============================================================
# PASO 1: Instalación de librerías necesarias
# - transformers: biblioteca de HuggingFace para modelos NLP
# - torch: PyTorch, el framework de deep learning
# - pandas: para mostrar resultados en tabla
# ============================================================

!pip install transformers torch pandas --quiet

print("✅ Librerías instaladas correctamente.")

✅ Librerías instaladas correctamente.


---
## 📦 PASO 2: Importación de módulos

In [2]:
# ============================================================
# PASO 2: Importamos los módulos necesarios
# - AutoTokenizer: tokenizador automático según el modelo
# - AutoModel: carga el modelo Transformer (sin cabeza de clasificación)
# - pipeline: forma simplificada de usar modelos de HuggingFace
# - torch: para operaciones tensoriales (matrices numéricas)
# - torch.nn.functional: funciones como softmax
# - pandas: para mostrar tabla de resultados
# ============================================================

import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("✅ Módulos importados correctamente.")
print(f"   PyTorch version: {torch.__version__}")

✅ Módulos importados correctamente.
   PyTorch version: 2.10.0+cpu


---
## 🤖 PASO 3: Carga del modelo preentrenado

Usamos **`nlptown/bert-base-multilingual-uncased-sentiment`**, un modelo BERT multilingüe preentrenado específicamente para análisis de sentimientos. Funciona con textos en español, inglés, francés, alemán, holandés e italiano.

In [3]:
# ============================================================
# PASO 3: Carga del modelo preentrenado desde HuggingFace Hub
#
# Modelo elegido: nlptown/bert-base-multilingual-uncased-sentiment
# - Basado en BERT multilingüe
# - Preentrenado para análisis de sentimientos
# - Soporte para español (ideal para nuestros textos de ejemplo)
# - Salida original: 1 a 5 estrellas (la adaptamos a Pos/Neg)
#
# AutoTokenizer: convierte texto en tokens que el modelo entiende
# AutoModelForSequenceClassification: modelo BERT con capa de clasificación
# ============================================================

NOMBRE_MODELO = "nlptown/bert-base-multilingual-uncased-sentiment"

print(f"⏳ Cargando modelo: {NOMBRE_MODELO}")
print("   (Primera vez puede tardar ~1 minuto, descarga ~700MB)")

# Cargamos el tokenizador asociado al modelo
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

# Cargamos el modelo con su capa de clasificación incluida
modelo = AutoModelForSequenceClassification.from_pretrained(NOMBRE_MODELO)

# Ponemos el modelo en modo evaluación (desactiva dropout)
modelo.eval()

print(f"\n✅ Modelo cargado correctamente.")
print(f"   Número de parámetros: {sum(p.numel() for p in modelo.parameters()):,}")
print(f"   Clases del modelo: {modelo.config.id2label}")

⏳ Cargando modelo: nlptown/bert-base-multilingual-uncased-sentiment
   (Primera vez puede tardar ~1 minuto, descarga ~700MB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ Modelo cargado correctamente.
   Número de parámetros: 167,360,261
   Clases del modelo: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}


---
## 📝 PASO 4: Textos de ejemplo a clasificar

In [4]:
# ============================================================
# PASO 4: Lista de textos de ejemplo
# Estos son los textos que el sistema analizará.
# Podés agregar o modificar los textos según tu necesidad.
# ============================================================

textos_ejemplo = [
    "La película fue increíble y me gustó mucho.",
    "El sistema funciona muy mal y presenta muchos errores.",
    "Estoy feliz porque finalicé mi proyecto.",
    "La experiencia fue decepcionante.",
    "El servicio fue excelente."
]

print("📋 Textos a analizar:")
for i, texto in enumerate(textos_ejemplo, 1):
    print(f"   {i}. {texto}")

📋 Textos a analizar:
   1. La película fue increíble y me gustó mucho.
   2. El sistema funciona muy mal y presenta muchos errores.
   3. Estoy feliz porque finalicé mi proyecto.
   4. La experiencia fue decepcionante.
   5. El servicio fue excelente.


---
## 🔬 PASO 5: Función de clasificación

Esta función recorre todo el flujo:
`Texto → Tokenización → Embedding → Atención Q,K,V → Promedio ponderado → Capa lineal → Positivo/Negativo`

In [5]:
# ============================================================
# PASO 5: Función principal de clasificación de sentimientos
#
# Esta función implementa el flujo completo:
#   1. Tokenización del texto
#   2. Generación de embeddings
#   3. Atención Q, K, V (internamente en el Transformer)
#   4. Promedio ponderado (mean pooling)
#   5. Capa lineal → logits
#   6. Softmax → probabilidades
#   7. Clasificación final: POSITIVO / NEGATIVO
# ============================================================

def clasificar_sentimiento(texto, tokenizador, modelo):
    """
    Clasifica el sentimiento de un texto usando un modelo Transformer.

    Args:
        texto (str): El texto a analizar.
        tokenizador: Tokenizador del modelo.
        modelo: Modelo Transformer con capa de clasificación.

    Returns:
        dict con: texto, cantidad_tokens, sentimiento, confianza
    """

    # ----------------------------------------------------------
    # TOKENIZACIÓN
    # El tokenizador convierte el texto en:
    #   - input_ids: lista de números (ID de cada token)
    #   - attention_mask: 1 = token real, 0 = padding
    # max_length=512 es el límite de BERT
    # return_tensors="pt" devuelve tensores de PyTorch
    # ----------------------------------------------------------
    tokens = tokenizador(
        texto,
        return_tensors="pt",    # formato PyTorch
        truncation=True,        # cortar si excede max_length
        max_length=512,         # máximo de tokens permitidos
        padding=True            # rellenar si es necesario
    )

    # Cantidad de tokens reales (sin contar padding)
    # input_ids[0] es la primera (y única) oración del batch
    cantidad_tokens = tokens["input_ids"].shape[1]

    # ----------------------------------------------------------
    # PROPAGACIÓN HACIA ADELANTE (Forward Pass)
    # torch.no_grad(): no calculamos gradientes (modo inferencia)
    #                  ahorra memoria y acelera el proceso
    #
    # Internamente el modelo realiza:
    #   1. Embedding: IDs → vectores de 768 dimensiones
    #   2. Positional Encoding: suma información de posición
    #   3. Capas de Atención Q,K,V (12 capas en BERT-base)
    #   4. Capa lineal final → logits (puntuaciones por clase)
    # ----------------------------------------------------------
    with torch.no_grad():
        salida = modelo(**tokens)

    # salida.logits: tensor de forma [1, num_clases]
    # Para este modelo: [1, 5] → 5 estrellas (1 a 5)
    logits = salida.logits

    # ----------------------------------------------------------
    # SOFTMAX: convertimos logits en probabilidades
    # Softmax asegura que todas las probabilidades sumen 1.0
    # Fórmula: softmax(x_i) = exp(x_i) / sum(exp(x_j))
    # dim=1 indica que operamos sobre la dimensión de clases
    # ----------------------------------------------------------
    probabilidades = F.softmax(logits, dim=1)

    # ----------------------------------------------------------
    # PROMEDIO PONDERADO (MEAN POOLING)
    # El modelo tiene 5 clases (1 a 5 estrellas).
    # Agrupamos:
    #   - NEGATIVO: clases 0 y 1 (1★ y 2★)
    #   - POSITIVO: clases 3 y 4 (4★ y 5★)
    #   - NEUTRAL:  clase 2 (3★) — la asignamos al más probable
    #
    # Esto implementa el "promedio ponderado" donde cada clase
    # aporta con su probabilidad al resultado final.
    # ----------------------------------------------------------
    probs = probabilidades[0].tolist()  # convertir a lista Python

    # Agrupamos probabilidades por polaridad
    prob_negativo = probs[0] + probs[1]   # clases 1★ y 2★
    prob_neutral  = probs[2]               # clase 3★
    prob_positivo = probs[3] + probs[4]   # clases 4★ y 5★

    # La neutral la asignamos al grupo con mayor probabilidad
    if prob_neutral > 0:
        if prob_positivo >= prob_negativo:
            prob_positivo += prob_neutral
        else:
            prob_negativo += prob_neutral

    # ----------------------------------------------------------
    # CLASIFICACIÓN FINAL
    # Comparamos las probabilidades agrupadas y decidimos
    # ----------------------------------------------------------
    if prob_positivo >= prob_negativo:
        sentimiento = "😊 POSITIVO"
        confianza = prob_positivo
    else:
        sentimiento = "😞 NEGATIVO"
        confianza = prob_negativo

    return {
        "texto": texto,
        "cantidad_tokens": cantidad_tokens,
        "sentimiento": sentimiento,
        "confianza": round(confianza * 100, 2)  # en porcentaje
    }


print("✅ Función de clasificación definida correctamente.")

✅ Función de clasificación definida correctamente.


---
## 🚀 PASO 6: Ejecutar la clasificación

In [6]:
# ============================================================
# PASO 6: Procesamos todos los textos de ejemplo
# Iteramos sobre cada texto, lo clasificamos y guardamos
# los resultados en una lista para luego mostrar en tabla.
# ============================================================

resultados = []  # lista para almacenar todos los resultados

print("⏳ Clasificando textos...\n")
print("-" * 60)

for i, texto in enumerate(textos_ejemplo, 1):
    # Llamamos a la función de clasificación para cada texto
    resultado = clasificar_sentimiento(texto, tokenizador, modelo)
    resultados.append(resultado)

    # Mostramos resultado parcial mientras procesa
    print(f"Texto {i}: \"{texto[:45]}...\"" if len(texto) > 45 else f"Texto {i}: \"{texto}\"")
    print(f"   → {resultado['sentimiento']} | Confianza: {resultado['confianza']}% | Tokens: {resultado['cantidad_tokens']}")
    print()

print("-" * 60)
print("✅ Clasificación completada.")

⏳ Clasificando textos...

------------------------------------------------------------
Texto 1: "La película fue increíble y me gustó mucho."
   → 😊 POSITIVO | Confianza: 98.93% | Tokens: 13

Texto 2: "El sistema funciona muy mal y presenta muchos..."
   → 😞 NEGATIVO | Confianza: 99.87% | Tokens: 13

Texto 3: "Estoy feliz porque finalicé mi proyecto."
   → 😊 POSITIVO | Confianza: 98.56% | Tokens: 11

Texto 4: "La experiencia fue decepcionante."
   → 😞 NEGATIVO | Confianza: 99.64% | Tokens: 10

Texto 5: "El servicio fue excelente."
   → 😊 POSITIVO | Confianza: 98.8% | Tokens: 7

------------------------------------------------------------
✅ Clasificación completada.


---
## 📊 PASO 7: Tabla de resultados

In [7]:
# ============================================================
# PASO 7: Mostrar resultados en una tabla formateada
# Usamos pandas DataFrame para una presentación clara.
# ============================================================

# Creamos el DataFrame con los resultados
df_resultados = pd.DataFrame(resultados)

# Renombramos las columnas para mejor presentación
df_resultados.columns = [
    "Texto analizado",
    "Tokens",
    "Sentimiento detectado",
    "Confianza (%)"
]

# Ajustamos el índice para que empiece en 1
df_resultados.index = range(1, len(df_resultados) + 1)
df_resultados.index.name = "#"

print("\n" + "=" * 70)
print("           📊 RESULTADOS DE CLASIFICACIÓN DE SENTIMIENTOS")
print("=" * 70)

# Mostramos la tabla con opciones de visualización
pd.set_option("display.max_colwidth", 50)   # ancho máximo de columna
pd.set_option("display.colheader_justify", "center")

display(df_resultados)


           📊 RESULTADOS DE CLASIFICACIÓN DE SENTIMIENTOS


,Texto analizado,Tokens,Sentimiento detectado,Confianza (%)
#,,,,
1,La película fue increíble y me gustó mucho.,13,😊 POSITIVO,98.93
2,El sistema funciona muy mal y presenta muchos ...,13,😞 NEGATIVO,99.87
3,Estoy feliz porque finalicé mi proyecto.,11,😊 POSITIVO,98.56
4,La experiencia fue decepcionante.,10,😞 NEGATIVO,99.64
5,El servicio fue excelente.,7,😊 POSITIVO,98.80


---
## 🔎 PASO 8 (EXTRA): Visualizar el proceso de tokenización

In [8]:
# ============================================================
# PASO 8 (EXTRA): Explorar el proceso de tokenización
# Aquí mostramos qué pasa internamente cuando tokenizamos.
# Esto ayuda a entender cómo el modelo "ve" el texto.
# ============================================================

texto_demo = "La película fue increíble y me gustó mucho."

print("=" * 60)
print("🔬 EXPLORACIÓN DEL PROCESO DE TOKENIZACIÓN")
print("=" * 60)
print(f"\nTexto original: '{texto_demo}'")
print()

# Tokenizamos el texto de demostración
tokens_demo = tokenizador(
    texto_demo,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Obtenemos los IDs de los tokens
input_ids = tokens_demo["input_ids"][0].tolist()

# Convertimos los IDs de vuelta a palabras/subpalabras
tokens_palabras = tokenizador.convert_ids_to_tokens(input_ids)

print("Tokens generados (subpalabras):")
print("-" * 40)
for i, (token_id, token_word) in enumerate(zip(input_ids, tokens_palabras)):
    # [CLS] = token especial de inicio
    # [SEP] = token especial de fin
    # ## = continuación de subpalabra
    nota = ""
    if token_word == "[CLS]":
        nota = " ← token especial de INICIO"
    elif token_word == "[SEP]":
        nota = " ← token especial de FIN"
    elif token_word.startswith("##"):
        nota = " ← subpalabra (continuación)"

    print(f"  Pos {i:2d} | ID: {token_id:6d} | Token: '{token_word}'{nota}")

print()
print(f"Total de tokens: {len(input_ids)}")
print()
print("💡 Nota: BERT agrega [CLS] al inicio y [SEP] al final.")
print("   Algunas palabras se dividen en subpalabras (## = continuación).")

🔬 EXPLORACIÓN DEL PROCESO DE TOKENIZACIÓN

Texto original: 'La película fue increíble y me gustó mucho.'

Tokens generados (subpalabras):
----------------------------------------
  Pos  0 | ID:    101 | Token: '[CLS]' ← token especial de INICIO
  Pos  1 | ID:  10106 | Token: 'la'
  Pos  2 | ID:  15141 | Token: 'pelicula'
  Pos  3 | ID:  10565 | Token: 'fue'
  Pos  4 | ID:  13565 | Token: 'inc'
  Pos  5 | ID:  31261 | Token: '##rei' ← subpalabra (continuación)
  Pos  6 | ID:  11522 | Token: '##ble' ← subpalabra (continuación)
  Pos  7 | ID:    167 | Token: 'y'
  Pos  8 | ID:  10525 | Token: 'me'
  Pos  9 | ID:  56303 | Token: 'gusto'
  Pos 10 | ID:  23505 | Token: 'mucho'
  Pos 11 | ID:    119 | Token: '.'
  Pos 12 | ID:    102 | Token: '[SEP]' ← token especial de FIN

Total de tokens: 13

💡 Nota: BERT agrega [CLS] al inicio y [SEP] al final.
   Algunas palabras se dividen en subpalabras (## = continuación).


---
## 🧪 PASO 9 (EXTRA): Probar con texto personalizado

In [9]:
# ============================================================
# PASO 9 (EXTRA): Ingresá tu propio texto y analizalo
# Modificá la variable 'mi_texto' con cualquier oración
# ============================================================

# 👇 CAMBIÁ ESTE TEXTO POR EL QUE QUIERAS PROBAR
mi_texto = "Me encantó la clase de Inteligencia Artificial, aprendí muchísimo."

# Clasificamos el texto personalizado
resultado_personalizado = clasificar_sentimiento(mi_texto, tokenizador, modelo)

print("=" * 60)
print("🧪 ANÁLISIS DE TEXTO PERSONALIZADO")
print("=" * 60)
print(f"\n📝 Texto:       '{mi_texto}'")
print(f"🔢 Tokens:      {resultado_personalizado['cantidad_tokens']}")
print(f"💬 Sentimiento: {resultado_personalizado['sentimiento']}")
print(f"📊 Confianza:   {resultado_personalizado['confianza']}%")

🧪 ANÁLISIS DE TEXTO PERSONALIZADO

📝 Texto:       'Me encantó la clase de Inteligencia Artificial, aprendí muchísimo.'
🔢 Tokens:      18
💬 Sentimiento: 😊 POSITIVO
📊 Confianza:   99.71%


---
## 📌 Resumen del Proyecto

| Componente | Detalle |
|---|---|
| **Modelo** | `nlptown/bert-base-multilingual-uncased-sentiment` |
| **Arquitectura** | BERT-base (12 capas, 768 dimensiones, 110M parámetros) |
| **Tokenizador** | WordPiece (divide palabras en subpalabras) |
| **Tarea** | Clasificación de sentimientos (Positivo / Negativo) |
| **Idiomas** | Español, inglés, francés, alemán, holandés, italiano |
| **Librería** | HuggingFace Transformers + PyTorch |

### ¿Qué aprendimos?
1. Los **Transformers** procesan texto completo en paralelo usando atención Q, K, V.
2. Los **embeddings** representan cada token como un vector en espacio numérico.
3. El **promedio ponderado** (mean pooling) resume todos los tokens en un solo vector.
4. La **capa lineal** transforma ese vector en puntuaciones por clase.
5. El **softmax** convierte las puntuaciones en probabilidades interpretables.

### Referencias
- Devlin et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- Vaswani et al. (2017). *Attention is All You Need*. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- HuggingFace Transformers: [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers)